# Explore a Document Collection

Every retrieval system starts with a **collection** — the set of documents we want
to search. This notebook lets you load any collection, inspect its structure,
and observe basic text statistics before applying retrieval models.

**Learning goals:**
- Understand the standard document structure: `id`, `source`, `text`, + metadata
- See how different collections split source material into retrieval units
- Observe Zipf's law and term statistics on real (and synthetic) text

In [ ]:
from shared.collections import load_collection, available_collections
from shared.display import print_table, display_md
from shared.text import tokenize, remove_stopwords, pipeline
from collections import Counter
import matplotlib.pyplot as plt
import ipywidgets as widgets

## Available Collections

In [ ]:
# List all registered collections from the registry
print_table(
    [[c["name"], c["description"]] for c in available_collections()],
    headers=["Name", "Description"]
)

## Select a Collection

Pick a collection for the rest of this notebook. Each collection handles its
own data — downloading, caching, and splitting into retrieval units automatically.

In [ ]:
# Build dropdown from the registry — every collection is a first-class entry
_names = [c["name"] for c in available_collections()]

_dropdown = widgets.Dropdown(
    options=_names,
    value="slides-classical-text-retrieval",
    description="Collection:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)
display(_dropdown)

In [ ]:
# Load the selected collection
collection = load_collection(_dropdown.value)

display_md(
    f"**{collection.name}** — {collection.description}\n\n"
    f"- Documents: **{len(collection)}** retrieval units\n"
    f"- Metadata keys: {collection.metadata_keys()}"
)

## Inspect the Documents

Each retrieval unit is a flat dictionary with at least `id`, `source`, and `text`.

In [ ]:
# Show the first 10 documents
rows = []
for doc in list(collection)[:10]:
    rows.append([
        doc["id"],
        doc["source"],
        doc["text"][:80] + ("..." if len(doc["text"]) > 80 else ""),
    ])

print_table(rows, headers=["ID", "Source", "Text (preview)"])

In [ ]:
# Look at a random document in full (re-run this cell to see another)
import random

doc = random.choice(collection.documents())
meta_lines = "\n".join(f"- **{key}:** {doc.get(key, '—')}" for key in collection.metadata_keys())

display_md(
    f"### Document `{doc['id']}`\n\n"
    f"- **Source:** {doc['source']}\n"
    f"{meta_lines}\n\n"
    f"---\n\n"
    f"{doc['text']}"
)

---
## Collection Statistics

How long are the documents? This determines how meaningful term frequencies are
and how well IDF can discriminate.

In [ ]:
# Document lengths (in tokens, after basic tokenization)
lengths = []
for doc in collection:
    tokens = tokenize(doc["text"])
    lengths.append(len(tokens))

display_md(
    f"**Token counts per document:**\n\n"
    f"| Statistic | Value |\n"
    f"|-----------|-------|\n"
    f"| Documents | {len(lengths)} |\n"
    f"| Total tokens | {sum(lengths):,} |\n"
    f"| Mean length | {sum(lengths)/len(lengths):.1f} tokens |\n"
    f"| Shortest | {min(lengths)} tokens |\n"
    f"| Longest | {max(lengths)} tokens |"
)

In [ ]:
# Distribution of document lengths
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(lengths)), sorted(lengths, reverse=True), color="steelblue", alpha=0.7)
ax.set_xlabel("Document (sorted by length)")
ax.set_ylabel("Tokens")
ax.set_title(f"Document Length Distribution — {collection.name}")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---
## Vocabulary and Term Frequencies

We tokenize the entire collection and look at the vocabulary.

In [ ]:
# Tokenize all documents (lowercase, split, remove stopwords)
all_tokens = []
doc_tokens = {}
for doc in collection:
    tokens = remove_stopwords(tokenize(doc["text"]), collection.language or "all")
    doc_tokens[doc["id"]] = tokens
    all_tokens.extend(tokens)

# Collection frequency (total occurrences)
cf = Counter(all_tokens)

# Document frequency (how many docs contain each term)
df = Counter()
for tokens in doc_tokens.values():
    df.update(set(tokens))

vocab = sorted(df.keys())

display_md(
    f"**Vocabulary:** {len(vocab):,} unique terms | "
    f"**Total tokens:** {len(all_tokens):,} (after stop word removal)"
)

In [ ]:
# Most frequent terms
print_table(
    [[term, count, df[term], f"{df[term]/len(collection)*100:.0f}%"]
     for term, count in cf.most_common(20)],
    headers=["Term", "Collection Freq.", "Doc. Freq.", "% docs"]
)

In [ ]:
# Rarest terms (appear in only 1 document)
rare = [t for t, f in df.items() if f == 1]
display_md(
    f"**Rare terms** (df=1): {len(rare)} / {len(vocab)} "
    f"({len(rare)/len(vocab)*100:.0f}% of vocabulary)\n\n"
    f"Examples: {', '.join(sorted(rare)[:20])}"
)

---
## Zipf's Law

The frequency distribution of terms in natural language follows a power law.
On a log-log plot, this appears as a straight line.

In [ ]:
# Sort by frequency
cf_sorted = cf.most_common()
ranks = range(1, len(cf_sorted) + 1)
freqs = [f for _, f in cf_sorted]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Linear scale
ax1.bar(range(len(freqs[:50])), freqs[:50], color="steelblue", alpha=0.7)
ax1.set_xlabel("Term rank")
ax1.set_ylabel("Collection frequency")
ax1.set_title("Top 50 Terms")

# Log-log (Zipf)
ax2.loglog(ranks, freqs, "o", markersize=2, color="steelblue", alpha=0.7)
ax2.set_xlabel("Rank (log)")
ax2.set_ylabel("Frequency (log)")
ax2.set_title("Zipf's Law")
ax2.grid(True, alpha=0.3)

plt.suptitle(f"Term Frequency Distribution — {collection.name}", y=1.02)
plt.tight_layout()
plt.show()

---
## Try It Yourself

Go back to the dropdown above and select a different collection — then
re-run the cells below it to see how statistics change.

Things to explore:
- How does MINI compare to a real lecture? (Select "mini" above)
- What happens when you load ALL slides? (Select "slides (all)")
- Which lecture has the richest vocabulary?

In [ ]:
# You can also load collections programmatically:
#
#   load_collection("slides", pdf="semantic-search")
#   load_collection("slides", pdf="all")
#   load_collection("slides", pdf="https://example.com/custom.pdf")
#   load_collection("mini")